# 02 — Shadow Comparison and the Migration Readiness Scorecard

Both backends on the same inputs: how often they agreed, who won where they did not,
and what the pre-agreed gates say about it.

Same rule as notebook 01 — this notebook displays, it does not compute.

In [ ]:
# papermill parameters
MODE = "replay"
CUSTOMER = "demo_patents"
SHADOW_METRIC = "structured"   # structured | item — see section 1

In [ ]:
%matplotlib inline

from IPython.display import Markdown, display

from amw.config import load_all
from amw.reporting import notebook as nb

cfg = load_all(customer=CUSTOMER)
shadow = nb.load_shadow_result()
phase2 = nb.load_phase2()
display(Markdown(f"**{nb.replay_banner(shadow)}**"))

## 1. Agreement — and which agreement

"Agreement" is not one number, and the difference is not cosmetic:

- **structured** — only the fields with a defined right answer. Prose is excluded and
  handled by the triage below.
- **item** — an item agrees only if *every* field matches, with prose scored by a
  token-overlap proxy at a fixed threshold. This figure moves substantially with that
  threshold, so it is partly a measurement of the instrument.

On this corpus they differ by up to 8x. The `shadow_agreement` gate is checked
against whichever one `SHADOW_METRIC` names, so the choice is a parameter, visible at
the top of this notebook, rather than a default buried in a function.

In [ ]:
agreement = nb.agreement_frame(shadow, metric=SHADOW_METRIC)
display(agreement)

fig = nb.interval_chart(
    agreement,
    label="subagent",
    point=f"{SHADOW_METRIC}_point",
    lo=f"{SHADOW_METRIC}_lo",
    hi=f"{SHADOW_METRIC}_hi",
    title=f"shadow_agreement ({SHADOW_METRIC}) — 95% CI, gate bound marked",
    bound=cfg.gates.subagent_gates["shadow_agreement"].min,
    bound_label="gate",
)
display(fig)

## 2. Latency — not a comparison

Claude ran in `global` (the us-central1 partner quota was exhausted); Gemini and the
judge ran in `us-central1`. These percentiles are real measurements of two different
regions, so they answer "how fast was each run" and not "which model is faster". The
`latency_p95` gate is reported as **not evaluated** for exactly this reason.

In [ ]:
latency = nb.latency_frame(shadow)
display(latency[nb.LATENCY_COLUMNS])
display(Markdown(f"> {latency['disclosure'].iloc[0]}"))

## 3. Disagreement triage

Every disagreement, adjudicated from the judge calls phase 2 already recorded — no
new judge call is issued here.

`not_adjudicated` is its own verdict with its own reason. An item outside the judged
split has no recorded verdict; calling that a tie would manufacture an outcome nobody
measured.

In [ ]:
for record in shadow.subagents:
    s = record.triage_summary
    print(
        f"{record.subagent}: {s.disagreements} disagreements — "
        f"{s.wins} win / {s.losses} loss / {s.ties} tie / {s.not_adjudicated} not adjudicated"
    )

In [ ]:
BROWSE_SUBAGENT = "query_rewriter"   # change and re-run to browse another
triage = nb.triage_frame(shadow, subagent=BROWSE_SUBAGENT)
display(triage["verdict"].value_counts().to_frame("rows"))
display(triage[triage["verdict"].isin(["win", "loss"])][nb.TRIAGE_COLUMNS])

## 4. The scorecard

Gates checked on their **CI lower bound**, then the verdict rules from
`config/gates.yaml`. A gate that was not measured is not a gate that passed: where
gates are missing the verdict is **INCOMPLETE**, with the verdict it *would* be
shown alongside and labelled as conditional.

In [ ]:
from amw.reporting import build_scorecard, render_markdown
from amw.reporting.scorecard import load_shadow

card = build_scorecard(
    cfg,
    phase2,
    shadow=load_shadow(nb.ARTIFACTS / "shadow.json", SHADOW_METRIC),
    shadow_metric=SHADOW_METRIC,
)
display(Markdown(render_markdown(card)))

### Reproducing this outside the notebook

```bash
python cli.py shadow --mode replay
python cli.py scorecard --mode replay \
    --shadow artifacts/results/shadow.json \
    --shadow-metric structured \
    --out artifacts/reports/scorecard.md
```

The scorecard above is built with `samples=None`, so the two paired-delta gates
(`quality_delta_pp`, `groundedness_delta_pp`) report as not evaluated here. The CLI
re-derives them by replaying the recorded calls; that takes about a minute, which is
why it is a command and not a cell.

In [ ]:
## 5. Vertex Gen AI Evaluation Service — a managed second instrument
#
# Everything above is measured by our harness with our judge and our rubrics. This
# section re-scores the *same recorded outputs* with Google's managed evaluation
# service: `general_quality_v1` over our own rubric group, `tool_use_quality_v1`
# over the recorded tool calls, and its loss clustering over the failures it found.
#
# The two sets of numbers sit side by side and are never merged — every row names
# the instrument that produced it. The gates are checked against the internal
# harness alone; nothing here feeds the scorecard. If the service was not reachable
# the banner says so and no managed figure appears, rather than a zero.
from IPython.display import Markdown, display

from amw.eval import vertex_eval as ve

managed = ve.load_result()
display(Markdown(f"**{ve.managed_banner(managed)}**"))
display(ve.side_by_side_frame(managed))
display(Markdown("**Loss clusters** — managed service; an arm with no clusters keeps its row."))
display(ve.loss_cluster_frame(managed))